# IOAI — 2025 Selection Camp Skeleton Action (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request
os.makedirs('data/seq', exist_ok=True)
_base = 'https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/skeletons-seq'
for f in ['train.pt', 'val.pt', 'test.pt']:
    if not os.path.exists(f'data/seq/{f}'):
        urllib.request.urlretrieve(f'{_base}/{f}', f'data/seq/{f}')
print('데이터 준비:', 'data/seq/train.pt' if os.path.exists('data/seq/train.pt') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Skeletons — 모범답안 (LSTM 시퀀스 분류)

> 데이터는 DGX에 **사전 준비**돼 있습니다 — `data/seq/{train,val,test}.pt`(IDSample별 가변길이 골격 시퀀스 `[T,75]`(25관절×XYZ) + train/val 의 Action·Camera 라벨). 느린 CSV 그룹화 없이 바로 학습합니다.

**과제**: 골격 관절좌표 시퀀스로 **Action(5클래스)**·**Camera(3클래스)** 를 분류. 지표 = 두 정확도 평균 **mean_acc**(높을수록 좋음). held-out val(IDSample%5==0)로 채점, 제출 `submission.csv`(datapointID, action, camera).

원대회 reference 방식: **Action·Camera 각각 LSTM**(2층, hidden 128) 으로 가변길이 시퀀스를 처리(packed sequence). 시간 동역학을 포착해 **Camera 정확도를 크게 올립니다**(베이스라인 mean≈0.74 → 모범답안≈0.80). 예측을 `submission.csv`(채점용) + `submission_judge.csv`(원대회 3-subtask 형식)로 저장.

In [ ]:
import torch, torch.nn as nn, numpy as np, pandas as pd
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tr = torch.load('data/seq/train.pt'); va = torch.load('data/seq/val.pt'); te = torch.load('data/seq/test.pt')

In [ ]:
class SeqDS(Dataset):
    def __init__(self, d, lab=None, off=0): self.d=d; self.lab=lab; self.off=off
    def __len__(self): return len(self.d['ids'])
    def __getitem__(self, i):
        s=self.d['seqs'][i]
        return (s, self.d[self.lab][i]-self.off) if self.lab else s
def coll(b):
    if isinstance(b[0], tuple):
        seqs,ys=zip(*b); return pad_sequence(seqs,batch_first=True), torch.tensor([len(x) for x in seqs]), torch.tensor(ys)
    return pad_sequence(b,batch_first=True), torch.tensor([len(x) for x in b])
class SkeletonNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__(); self.lstm=nn.LSTM(75,128,2,batch_first=True); self.head=nn.Linear(128,num_classes)
    def forward(self, x, L):
        p=pack_padded_sequence(x, L.cpu(), batch_first=True, enforce_sorted=False)
        _,(h,_)=self.lstm(p); return self.head(h[-1])

In [ ]:
def train_model(lab, nclass, off, epochs=30):
    torch.manual_seed(333)
    dl=DataLoader(SeqDS(tr,lab,off), batch_size=32, shuffle=True, collate_fn=coll, drop_last=True)
    m=SkeletonNet(nclass).to(dev); opt=torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4); crit=nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for X,L,y in dl: X,y=X.to(dev),y.to(dev); opt.zero_grad(); crit(m(X,L),y).backward(); opt.step()
    return m
@torch.no_grad()
def predict(m, d, off):
    m.eval(); dl=DataLoader(SeqDS(d), batch_size=64, shuffle=False, collate_fn=coll); out=[]
    for X,L in dl: out.extend((m(X.to(dev),L).argmax(1).cpu()+off).tolist())
    return out
m_act=train_model('action',5,0); m_cam=train_model('camera',3,1)   # camera 라벨 1-3 -> 0-2
va_act=predict(m_act,va,0); va_cam=predict(m_cam,va,1)
print('val action_acc', round(float(np.mean(np.array(va_act)==va['action'])),4), '| camera_acc', round(float(np.mean(np.array(va_cam)==va['camera'])),4))

In [ ]:
# held-out val 예측 → submission.csv (우리 채점기용)
pd.DataFrame({'datapointID': va['ids'], 'action': va_act, 'camera': va_cam}).to_csv('submission.csv', index=False)
# test 예측 → submission_judge.csv (원대회 3-subtask: 1=프레임수, 2=camera, 3=action)
te_act=predict(m_act,te,0); te_cam=predict(m_cam,te,1); te_frames=[len(s) for s in te['seqs']]
sub=pd.concat([
    pd.DataFrame({'datapointID':te['ids'],'answer':te_frames,'subtaskID':1}),
    pd.DataFrame({'datapointID':te['ids'],'answer':te_cam,'subtaskID':2}),
    pd.DataFrame({'datapointID':te['ids'],'answer':te_act,'subtaskID':3})])
sub.to_csv('submission_judge.csv', index=False)
print('wrote submission.csv (val) + submission_judge.csv (test)', len(te['ids']))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)